In [2]:
# 산탄데르 은행 고객 데이터 만족 예측 모델 
# 피처 : 370개
# 종속변수 : 1(불만족), 0(만족)
# 1. 무엇을 학습할 것인지(불균형 데이터 처리)
# 1-1 분산이 0인 데이터 변수 제거
# 중복된 컬럼 제거
# 이상치 처리


In [ ]:
# target = 종속변수
# var3 =지역
# var15 = 나이ㄴ
# imputed/Transaction Amount : 특정 기간 동안 발생한 거래 금액이나 금액 관련 지표
# operation : 은행 거래 횟수
# saldo/Balance : 계좌 잔액 관련 변수 
# saldo_medio : 특정 기간 동안 평균 잔액 
# var38 연속형 금융 변수, 대출액수

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import warnings
warnings.filterwarnings('ignore')


# 1. 데이터 로딩
cust_df = pd.read_csv("../data/train.csv", encoding='latin-1')
print('dataset shape:', cust_df.shape)
cust_df.head(10)

dataset shape: (76020, 371)


,ID,var3,var15,imp_ent_var16_ult1,imp_op_var39_comer_ult1,imp_op_var39_comer_ult3,imp_op_var40_comer_ult1,imp_op_var40_comer_ult3,imp_op_var40_efect_ult1,imp_op_var40_efect_ult3,...,saldo_medio_var33_hace2,saldo_medio_var33_hace3,saldo_medio_var33_ult1,saldo_medio_var33_ult3,saldo_medio_var44_hace2,saldo_medio_var44_hace3,saldo_medio_var44_ult1,saldo_medio_var44_ult3,var38,TARGET
0,1,2,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,39205.170000,0
1,3,2,34,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,49278.030000,0
2,4,2,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,67333.770000,0
3,8,2,37,0.0,195.0,195.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,64007.970000,0
4,10,2,39,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,117310.979016,0
5,13,2,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,87975.750000,0
6,14,2,27,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,94956.660000,0
7,18,2,26,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,251638.950000,0
8,20,2,45,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,101962.020000,0
9,23,2,25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,356463.060000,0


In [5]:
cust_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 76020 entries, 0 to 76019
Columns: 371 entries, ID to TARGET
dtypes: float64(111), int64(260)
memory usage: 215.2 MB


In [6]:
# 불균형 데이터 셋
# 분류 문제 첫번째 레이블(답) 불균형 확인

print(cust_df['TARGET'].value_counts())
unsatisfied_cnt = cust_df[cust_df['TARGET'] == 1].TARGET.count()
total_cnt = cust_df.TARGET.count()
print('unsatisfied 비율은 {0:.2f}'.format((unsatisfied_cnt / total_cnt)))

TARGET
0    73012
1     3008
Name: count, dtype: int64
unsatisfied 비율은 0.04


In [7]:
# 데이터 집계
cust_df.describe()
# var3 이상치 

,ID,var3,var15,imp_ent_var16_ult1,imp_op_var39_comer_ult1,imp_op_var39_comer_ult3,imp_op_var40_comer_ult1,imp_op_var40_comer_ult3,imp_op_var40_efect_ult1,imp_op_var40_efect_ult3,...,saldo_medio_var33_hace2,saldo_medio_var33_hace3,saldo_medio_var33_ult1,saldo_medio_var33_ult3,saldo_medio_var44_hace2,saldo_medio_var44_hace3,saldo_medio_var44_ult1,saldo_medio_var44_ult3,var38,TARGET
count,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,...,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,7.602000e+04,76020.000000
mean,75964.050723,-1523.199277,33.212865,86.208265,72.363067,119.529632,3.559130,6.472698,0.412946,0.567352,...,7.935824,1.365146,12.215580,8.784074,31.505324,1.858575,76.026165,56.614351,1.172358e+05,0.039569
std,43781.947379,39033.462364,12.956486,1614.757313,339.315831,546.266294,93.155749,153.737066,30.604864,36.513513,...,455.887218,113.959637,783.207399,538.439211,2013.125393,147.786584,4040.337842,2852.579397,1.826646e+05,0.194945
min,1.000000,-999999.000000,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,5.163750e+03,0.000000
25%,38104.750000,2.000000,23.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,6.787061e+04,0.000000
50%,76043.000000,2.000000,28.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.064092e+05,0.000000
75%,113748.750000,2.000000,40.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.187563e+05,0.000000
max,151838.000000,238.000000,105.000000,210000.000000,12888.030000,21024.810000,8237.820000,11073.570000,6600.000000,6600.000000,...,50003.880000,20385.720000,138831.630000,91778.730000,438329.220000,24650.010000,681462.900000,397884.300000,2.203474e+07,1.000000


In [8]:
cust_df['var3'].value_counts()
# var3 이상치 확인

var3
 2         74165
 8           138
-999999      116
 9           110
 3           108
           ...  
 63            1
 194           1
 40            1
 57            1
 87            1
Name: count, Length: 208, dtype: int64

In [9]:
# var3(결측치)평균대치, id 제거 (필요없음)
# Var3, id 
cust_df['var3'] = cust_df['var3'].replace(-999999, 2)
cust_df.drop('ID', axis=1, inplace=True)

In [10]:
# 정제데이터 확인
cust_df.describe()

,var3,var15,imp_ent_var16_ult1,imp_op_var39_comer_ult1,imp_op_var39_comer_ult3,imp_op_var40_comer_ult1,imp_op_var40_comer_ult3,imp_op_var40_efect_ult1,imp_op_var40_efect_ult3,imp_op_var40_ult1,...,saldo_medio_var33_hace2,saldo_medio_var33_hace3,saldo_medio_var33_ult1,saldo_medio_var33_ult3,saldo_medio_var44_hace2,saldo_medio_var44_hace3,saldo_medio_var44_ult1,saldo_medio_var44_ult3,var38,TARGET
count,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,...,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,7.602000e+04,76020.000000
mean,2.716483,33.212865,86.208265,72.363067,119.529632,3.559130,6.472698,0.412946,0.567352,3.160715,...,7.935824,1.365146,12.215580,8.784074,31.505324,1.858575,76.026165,56.614351,1.172358e+05,0.039569
std,9.447971,12.956486,1614.757313,339.315831,546.266294,93.155749,153.737066,30.604864,36.513513,95.268204,...,455.887218,113.959637,783.207399,538.439211,2013.125393,147.786584,4040.337842,2852.579397,1.826646e+05,0.194945
min,0.000000,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,5.163750e+03,0.000000
25%,2.000000,23.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,6.787061e+04,0.000000
50%,2.000000,28.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.064092e+05,0.000000
75%,2.000000,40.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.187563e+05,0.000000
max,238.000000,105.000000,210000.000000,12888.030000,21024.810000,8237.820000,11073.570000,6600.000000,6600.000000,8237.820000,...,50003.880000,20385.720000,138831.630000,91778.730000,438329.220000,24650.010000,681462.900000,397884.300000,2.203474e+07,1.000000


In [11]:
X_features = cust_df.iloc[:,:-1]
y_labels = cust_df.iloc[:,-1]
print(f'피처 데이터 shape은 : {X_features.shape}')

피처 데이터 shape은 : (76020, 369)


In [12]:
# 데이터 분리 
# 학습과 테스트 데이터 
# 분포 확인
# 학습/테스트 데이터 분리, 분포 확인
from sklearn.model_selection import train_test_split


X_train, X_test, y_train, y_test = train_test_split(X_features, y_labels,
                                                    test_size=0.2, random_state=0)
train_cnt = y_train.count()
test_cnt = y_test.count()
print('학습 세트 Shape:{0}, 테스트 세트 Shape:{1}'.format(X_train.shape , X_test.shape))


print(' 학습 세트 레이블 값 분포 비율')
print(y_train.value_counts()/train_cnt)
print('\n 테스트 세트 레이블 값 분포 비율')
print(y_test.value_counts()/test_cnt)

학습 세트 Shape:(60816, 369), 테스트 세트 Shape:(15204, 369)
 학습 세트 레이블 값 분포 비율
TARGET
0    0.960964
1    0.039036
Name: count, dtype: float64

 테스트 세트 레이블 값 분포 비율
TARGET
0    0.9583
1    0.0417
Name: count, dtype: float64


In [13]:
# 학습 데이터를 학습 데이터와 검증 데이터로 변환
# X_train, y_train을 다시 학습과 검증 데이터 세트로 분리.
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train,
                                                    test_size=0.3, random_state=0)


In [14]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score




# 1. 모델 선언
gb_clf = GradientBoostingClassifier(
    n_estimators=300, 
    learning_rate=0.05, 
    max_depth=4, 
    random_state=0
)
# 2. 검증 데이터로 예측 (확률값 산출)
gb_clf.fit(X_tr, y_tr)
y_val_pred_proba = gb_clf.predict_proba(X_val)[:, 1]

# 4. ROC-AUC 성능 평가
roc_score = roc_auc_score(y_val, y_val_pred_proba)
print(f"GBM 검증 세트 ROC-AUC Score: {roc_score:.4f}")

GBM 검증 세트 ROC-AUC Score: 0.8338


In [15]:
# 1. 각 컬럼의 분산 계산
stds = X_features.std()

# 2. 분산이 0인(즉, 표준편차가 0이거나 값이 모두 똑같은) 컬럼 이름 추출
zero_var_cols = stds[stds == 0].index.tolist()

print(f"분산이 0인 컬럼 개수: {len(zero_var_cols)}")
print(f"삭제할 컬럼들: {zero_var_cols}")

# 3. 해당 컬럼들 제거
X_features_clean = X_features.drop(columns=zero_var_cols)

print(f"정제 전 피처 shape: {X_features.shape}")
print(f"정제 후 피처 shape: {X_features_clean.shape}")

분산이 0인 컬럼 개수: 34
삭제할 컬럼들: ['ind_var2_0', 'ind_var2', 'ind_var27_0', 'ind_var28_0', 'ind_var28', 'ind_var27', 'ind_var41', 'ind_var46_0', 'ind_var46', 'num_var27_0', 'num_var28_0', 'num_var28', 'num_var27', 'num_var41', 'num_var46_0', 'num_var46', 'saldo_var28', 'saldo_var27', 'saldo_var41', 'saldo_var46', 'imp_amort_var18_hace3', 'imp_amort_var34_hace3', 'imp_reemb_var13_hace3', 'imp_reemb_var33_hace3', 'imp_trasp_var17_out_hace3', 'imp_trasp_var33_out_hace3', 'num_var2_0_ult1', 'num_var2_ult1', 'num_reemb_var13_hace3', 'num_reemb_var33_hace3', 'num_trasp_var17_out_hace3', 'num_trasp_var33_out_hace3', 'saldo_var2_ult1', 'saldo_medio_var13_medio_hace3']
정제 전 피처 shape: (76020, 369)
정제 후 피처 shape: (76020, 335)


In [16]:
# 1. 분산 0인 컬럼이 제거된 X_features_clean을 사용해 1차 분할 (Train / Test)
# 불균형 데이터이므로 stratify=y_labels 
X_train, X_test, y_train, y_test = train_test_split(
    X_features_clean, y_labels,
    test_size=0.2, 
    random_state=0,
    stratify=y_labels
)
# 2. 2차 분할: Train 세트를 다시 훈련용(Tr)과 검증용(Val)으로 분할 (Train / Validation)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train,
    test_size=0.3, 
    random_state=0,
    stratify=y_train
)
print(f"학습용(Tr) 데이터 Shape: {X_tr.shape}")
print(f"검증용(Val) 데이터 Shape: {X_val.shape}")

# 3. GBM 모델 선언
gb_clf = GradientBoostingClassifier(
    n_estimators=100,     # 속도를 고려해 우선 100개로 설정
    learning_rate=0.05, 
    max_depth=4, 
    random_state=0
)

# 4. 모델 학습 (fit)
gb_clf.fit(X_tr, y_tr)

# 5. 검증 데이터로 예측 (확률값 추출)
y_val_pred_proba = gb_clf.predict_proba(X_val)[:, 1]

# 6. ROC-AUC 성능 평가
roc_score = roc_auc_score(y_val, y_val_pred_proba)
print(f"GBM 검증 세트 ROC-AUC Score: {roc_score:.4f}")

학습용(Tr) 데이터 Shape: (42571, 335)
검증용(Val) 데이터 Shape: (18245, 335)
GBM 검증 세트 ROC-AUC Score: 0.8440


In [17]:
gb_clf = GradientBoostingClassifier(
    n_estimators=200,     # 속도를 고려해 우선 100개로 설정
    learning_rate=0.05, 
    max_depth=4, 
    random_state=0
)

# 4. 모델 학습 (fit)
gb_clf.fit(X_tr, y_tr)

# 5. 검증 데이터로 예측 (확률값 추출)
y_val_pred_proba = gb_clf.predict_proba(X_val)[:, 1]

# 6. ROC-AUC 성능 평가
roc_score = roc_auc_score(y_val, y_val_pred_proba)
print(f"GBM 검증 세트 ROC-AUC Score: {roc_score:.4f}")

GBM 검증 세트 ROC-AUC Score: 0.8465


In [18]:
gb_clf = GradientBoostingClassifier(
    n_estimators=300,     # 속도를 고려해 우선 100개로 설정
    learning_rate=0.05, 
    max_depth=4, 
    random_state=0
)

# 4. 모델 학습 (fit)
gb_clf.fit(X_tr, y_tr)

# 5. 검증 데이터로 예측 (확률값 추출)
y_val_pred_proba = gb_clf.predict_proba(X_val)[:, 1]

# 6. ROC-AUC 성능 평가
roc_score = roc_auc_score(y_val, y_val_pred_proba)
print(f"GBM 검증 세트 ROC-AUC Score: {roc_score:.4f}")

GBM 검증 세트 ROC-AUC Score: 0.8464
